# Stage 1 Interactive 3D Visualization (PyVista + Trame)

交互式渲染 FMT 重建结果：可在浏览器中旋转、缩放、调透明度，找到满意角度后一键截图导出论文图。

**依赖**（如未安装）：
```bash
pip install "pyvista[jupyter]" trame trame-vtk trame-vuetify jupyter ipywidgets
```

**运行**：
```bash
cd /home/foods/pro/DU2Vox
jupyter lab notebooks/visualize_stage1.ipynb
```

In [9]:
import numpy as np
import json
import torch
import yaml
from pathlib import Path
import pyvista as pv

# Enable Trame backend → embedded interactive 3D in Jupyter
pv.set_jupyter_backend("trame")
pv.global_theme.background = "white"
pv.global_theme.font.family = "arial"
pv.global_theme.font.size = 14
pv.global_theme.anti_aliasing = "ssaa"

In [10]:
# Load shared mesh data
shared_dir = Path("/home/foods/pro/FMT-SimGen/output/shared")

mesh_data = np.load(shared_dir / "mesh.npz", allow_pickle=True)
nodes = mesh_data["nodes"]                    # [N, 3]
elements = mesh_data["elements"]              # [~60K, 4]
surface_faces = mesh_data["surface_faces"]    # [~15K, 3]
tissue_labels = mesh_data["tissue_labels"]    # [~60K]
surface_node_indices = mesh_data["surface_node_indices"]

print(f"Nodes: {nodes.shape}, Elements: {elements.shape}")
print(f"Node range: X [{nodes[:,0].min():.1f}, {nodes[:,0].max():.1f}], "
      f"Y [{nodes[:,1].min():.1f}, {nodes[:,1].max():.1f}], "
      f"Z [{nodes[:,2].min():.1f}, {nodes[:,2].max():.1f}]")

Nodes: (11236, 3), Elements: (57730, 4)
Node range: X [2.4, 34.4], Y [4.8, 92.8], Z [1.6, 20.0]


In [11]:
def build_tet_grid(nodes, elements):
    """Build tetrahedral UnstructuredGrid from FEM data."""
    n_cells = len(elements)
    cells = np.column_stack([
        np.full(n_cells, 4, dtype=np.int64),
        elements.astype(np.int64)
    ]).ravel()
    cell_types = np.full(n_cells, pv.CellType.TETRA, dtype=np.uint8)
    return pv.UnstructuredGrid(cells, cell_types, nodes.astype(np.float64))

def build_surface(nodes, faces):
    """Build surface PolyData."""
    faces_pv = np.column_stack([np.full(len(faces), 3), faces]).ravel()
    return pv.PolyData(nodes, faces_pv)

# Build once (reusable for all samples)
tet_grid = build_tet_grid(nodes, elements)
body_surface = build_surface(nodes, surface_faces)
body_surface_smooth = body_surface.smooth(n_iter=50, relaxation_factor=0.1)

print(f"Tet grid: {tet_grid.n_cells} cells")
print(f"Body surface: {body_surface.n_points} points, {body_surface.n_cells} faces")

Tet grid: 57730 cells
Body surface: 11236 points, 33170 faces


In [12]:
# Extract organ surfaces from element-level tissue labels
# NOTE: Run "Inspect tissue labels" cell first to find correct label values for your mesh.
# Placeholder config — update with actual labels from your data.
ORGAN_CONFIG = {
    # label: (name, rgb_color, opacity)
    # Example (verify with your mesh):
    2:  ("Skeleton",  (0.95, 0.95, 0.90), 0.10),
    9:  ("Heart",     (0.80, 0.15, 0.15), 0.20),
    10: ("Liver",    (0.55, 0.15, 0.15), 0.18),
    11: ("Kidney",   (0.65, 0.30, 0.20), 0.20),
    5:  ("Lung",     (0.70, 0.85, 0.95), 0.15),
    6:  ("Spleen",   (0.60, 0.20, 0.30), 0.15),
}

organ_surfaces = {}
for label, (name, color, opacity) in ORGAN_CONFIG.items():
    mask = tissue_labels == label
    if mask.sum() < 10:
        continue
    organ_tet = tet_grid.extract_cells(np.where(mask)[0])
    organ_surf = organ_tet.extract_surface().smooth(n_iter=20, relaxation_factor=0.1)
    if organ_surf.n_points > 0:
        organ_surfaces[label] = organ_surf
        print(f"  {name}: {organ_surf.n_points} surface points")

print(f"Extracted {len(organ_surfaces)} organs")

  Skeleton: 2331 surface points
  Heart: 1017 surface points
  Liver: 304 surface points
  Kidney: 443 surface points
  Lung: 184 surface points
  Spleen: 182 surface points
Extracted 6 organs


/tmp/ipykernel_27970/2884384734.py:21: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  organ_surf = organ_tet.extract_surface().smooth(n_iter=20, relaxation_factor=0.1)
/tmp/ipykernel_27970/2884384734.py:21: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  organ_surf = organ_tet.extract_surface().smooth(n_iter=20, relaxation_factor=0.1)
/tmp/ipykernel_27970/2884384734.py:21: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`,

In [13]:
import sys
sys.path.insert(0, "/home/foods/pro/DU2Vox")
from du2vox.models.stage1.gcain import GCAIN_full
import scipy.sparse

# Load shared assets for model
A_sp = scipy.sparse.load_npz(shared_dir / "system_matrix.A.npz")
A = torch.tensor(A_sp.toarray(), dtype=torch.float32).cuda()

def load_npz_dense(path):
    return torch.tensor(scipy.sparse.load_npz(path).toarray(), dtype=torch.float32)

L  = load_npz_dense(shared_dir / "graph_laplacian_full.Lap.npz").cuda()
L0 = load_npz_dense(shared_dir / "graph_laplacian_full.n_Lap0.npz").cuda()
L1 = load_npz_dense(shared_dir / "graph_laplacian_full.n_Lap1.npz").cuda()
L2 = load_npz_dense(shared_dir / "graph_laplacian_full.n_Lap2.npz").cuda()
L3 = load_npz_dense(shared_dir / "graph_laplacian_full.n_Lap3.npz").cuda()
knn_idx = torch.tensor(np.load(shared_dir / "knn_idx_full.npy"), dtype=torch.long).cuda()
sens_w = torch.norm(A, dim=0)
sens_w = (sens_w / (sens_w.max() + 1e-8)).cuda()

n_nodes = nodes.shape[0]

def load_model(checkpoint_path, num_layer=6, feat_dim=6):
    model = GCAIN_full(
        L=L, A=A, L0=L0, L1=L1, L2=L2, L3=L3,
        knn_idx=knn_idx, sens_w=sens_w,
        num_layer=num_layer, feat_dim=feat_dim,
    ).cuda()
    ckpt = torch.load(checkpoint_path, map_location="cuda")
    if "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)
    model.eval()
    return model

model_g = load_model("/home/foods/pro/DU2Vox/runs/gcain_gaussian_1000/checkpoints/best.pth")
model_u = load_model("/home/foods/pro/DU2Vox/runs/gcain_uniform_1000/checkpoints/best.pth")
print("Models loaded")

Task was destroyed but it is pending!
task: <Task pending name='Task-3' coro=<_async_in_context.<locals>.run_in_context() running at /home/foods/pro/DU2Vox/.venv/lib/python3.12/site-packages/ipykernel/utils.py:60> wait_for=<Task pending name='Task-5' coro=<Kernel.shell_main() running at /home/foods/pro/DU2Vox/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/foods/pro/DU2Vox/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-2' coro=<Kernel.shell_main() running at /home/foods/pro/DU2Vox/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>
/home/foods/pro/DU2Vox/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  return torch._C._cuda_getDeviceCount() > 0
Task was destroyed but it is pending!
task:

Models loaded


In [14]:
def predict_sample(sample_dir, model, normalize_b=True,
                   normalize_gt=True, binarize_gt=False):
    """Load a sample and run inference. Returns (gt_nodes, pred_nodes, tumor_params)."""
    b = np.load(sample_dir / "measurement_b.npy")
    gt = np.load(sample_dir / "gt_nodes.npy")
    with open(sample_dir / "tumor_params.json") as f:
        params = json.load(f)

    # Normalize measurement
    b_t = torch.tensor(b, dtype=torch.float32).unsqueeze(-1)
    if normalize_b:
        b_max = b_t.max()
        if b_max > 1e-8:
            b_t = b_t / b_max

    # Process GT
    gt_out = gt.copy()
    if binarize_gt:
        gt_out = (gt > 0.05).astype(np.float32)
    elif normalize_gt:
        gt_max = gt.max()
        if gt_max > 1e-8:
            gt_out = gt / gt_max

    # Inference
    with torch.no_grad():
        b_cuda = b_t.unsqueeze(0).cuda()
        X0 = torch.zeros(1, n_nodes, 1, device="cuda")
        pred = model(X0, b_cuda)
        pred = torch.nn.functional.leaky_relu(pred, 0.01).clamp(max=1.0)

    pred_np = pred.squeeze().cpu().numpy()
    return gt_out, pred_np, params

# Sample paths
samples_g = Path("/home/foods/pro/FMT-SimGen/data/gaussian_1000/samples")
samples_u = Path("/home/foods/pro/FMT-SimGen/data/uniform_1000/samples")

# === Try a sample ===
sample_id = "sample_0042"
gt_g, pred_g, params_g = predict_sample(samples_g / sample_id, model_g)
gt_u, pred_u, params_u = predict_sample(
    samples_u / sample_id, model_u,
    normalize_gt=False, binarize_gt=False
)
print(f"Sample: {sample_id}")
print(f"  Gaussian GT: max={gt_g.max():.4f}, nonzero={np.count_nonzero(gt_g)}")
print(f"  Gaussian Pred: max={pred_g.max():.4f}")
print(f"  Uniform GT: max={gt_u.max():.4f}, nonzero={np.count_nonzero(gt_u)}")
print(f"  Uniform Pred: max={pred_u.max():.4f}")
print(f"  Foci: {params_g.get('num_foci', '?')}")

Sample: sample_0042
  Gaussian GT: max=1.0000, nonzero=1161
  Gaussian Pred: max=1.0000
  Uniform GT: max=1.0000, nonzero=137
  Uniform Pred: max=1.0000
  Foci: 2


In [15]:
def render_interactive(gt_vals, pred_vals, tumor_params,
                       mode="intensity",
                       gt_thresh=0.05, pred_thresh=0.3):
    """
    Interactive rendering with two synchronized sub-views (GT | Pred).
    - mode="intensity": shows YlOrRd heatmap of values above threshold
    - mode="segmentation": shows TP (green) / FP (blue) / FN (red)
    """
    pl = pv.Plotter(shape=(1, 2), window_size=(1600, 800))

    for col, (vals, title) in enumerate([
        (gt_vals, "Ground Truth"),
        (pred_vals, "Prediction")
    ]):
        pl.subplot(0, col)

        # Layer 1: semi-transparent body surface
        pl.add_mesh(body_surface_smooth, color=(0.92, 0.85, 0.78),
                    opacity=0.06, smooth_shading=True)

        # Layer 2: organs
        for label, surf in organ_surfaces.items():
            name, color, opacity = ORGAN_CONFIG[label]
            pl.add_mesh(surf, color=color, opacity=opacity,
                       smooth_shading=True)

        # Layer 3: tumor values
        if mode == "intensity":
            # Node values → element mean
            elem_vals = vals[elements].mean(axis=1)
            threshold = gt_thresh if col == 0 else pred_thresh
            active = elem_vals > threshold
            if active.any():
                active_tet = tet_grid.extract_cells(np.where(active)[0])
                active_tet.cell_data["intensity"] = elem_vals[active]
                active_surf = active_tet.extract_surface()
                pl.add_mesh(
                    active_surf,
                    scalars="intensity",
                    cmap="YlOrRd",
                    clim=[0, max(gt_vals.max(), 0.01)],
                    opacity=0.85,
                    smooth_shading=True,
                    scalar_bar_args={
                        "title": "Intensity",
                        "width": 0.05,
                        "position_x": 0.90
                    }
                )

        elif mode == "segmentation":
            elem_gt = (gt_vals[elements] > gt_thresh).sum(axis=1) >= 2
            elem_pred = (pred_vals[elements] > pred_thresh).sum(axis=1) >= 2

            if col == 0:
                # GT column: show GT mask only
                if elem_gt.any():
                    gt_tet = tet_grid.extract_cells(np.where(elem_gt)[0])
                    gt_surf = gt_tet.extract_surface()
                    pl.add_mesh(gt_surf, color=(0.9, 0.2, 0.1),
                               opacity=0.85, smooth_shading=True)
            else:
                # Pred column: TP / FP / FN
                tp = elem_gt & elem_pred
                fp = (~elem_gt) & elem_pred
                fn = elem_gt & (~elem_pred)
                if tp.any():
                    s = tet_grid.extract_cells(np.where(tp)[0]).extract_surface()
                    pl.add_mesh(s, color=(0.2, 0.8, 0.2), opacity=0.85,
                               smooth_shading=True, label="TP")
                if fp.any():
                    s = tet_grid.extract_cells(np.where(fp)[0]).extract_surface()
                    pl.add_mesh(s, color=(0.3, 0.5, 0.9), opacity=0.6,
                               smooth_shading=True, label="FP")
                if fn.any():
                    s = tet_grid.extract_cells(np.where(fn)[0]).extract_surface()
                    pl.add_mesh(s, color=(0.9, 0.2, 0.2), opacity=0.85,
                               smooth_shading=True, label="FN")
                pl.add_legend(bcolor="white", face="circle", size=(0.12, 0.10))

        # Layer 4: focus centre markers
        for focus in tumor_params.get("foci", []):
            center = np.array(focus["center"])
            sphere = pv.Sphere(radius=0.4, center=center)
            pl.add_mesh(sphere, color="yellow", opacity=0.95)

        pl.add_text(title, position="upper_left", font_size=14, color="black")
        pl.link_views()  # sync rotation across sub-views

    return pl

## Gaussian — Intensity View

In [16]:
# Rotate/zoom in the cell output, then go to Cell → Run All Below
pl = render_interactive(gt_g, pred_g, params_g,
                        mode="intensity",
                        gt_thresh=0.05, pred_thresh=0.1)
pl.show()

/tmp/ipykernel_27970/3669107851.py:36: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  active_surf = active_tet.extract_surface()
/tmp/ipykernel_27970/3669107851.py:36: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  active_surf = active_tet.extract_surface()


Widget(value='<iframe src="http://localhost:36181/index.html?ui=P_0x7f92d88c81d0_1&reconnect=auto" class="pyvi…

## Gaussian — Segmentation View (TP / FP / FN)

In [17]:
pl = render_interactive(gt_g, pred_g, params_g,
                        mode="segmentation",
                        gt_thresh=0.05, pred_thresh=0.3)
pl.show()

/tmp/ipykernel_27970/3669107851.py:59: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  gt_surf = gt_tet.extract_surface()
/tmp/ipykernel_27970/3669107851.py:68: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  s = tet_grid.extract_cells(np.where(tp)[0]).extract_surface()
/tmp/ipykernel_27970/3669107851.py:72: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword 

Widget(value='<iframe src="http://localhost:36181/index.html?ui=P_0x7f92d88a2d80_2&reconnect=auto" class="pyvi…

## Uniform — Segmentation View

Note: Uniform GT is binary (binarize_gt=true in config), so use threshold 0.5.

In [18]:
pl = render_interactive(gt_u, pred_u, params_u,
                        mode="segmentation",
                        gt_thresh=0.5, pred_thresh=0.5)
pl.show()

/tmp/ipykernel_27970/3669107851.py:59: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  gt_surf = gt_tet.extract_surface()
/tmp/ipykernel_27970/3669107851.py:68: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  s = tet_grid.extract_cells(np.where(tp)[0]).extract_surface()
/tmp/ipykernel_27970/3669107851.py:72: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword 

Widget(value='<iframe src="http://localhost:36181/index.html?ui=P_0x7f92d88a2f30_3&reconnect=auto" class="pyvi…

IOStream.flush timed out


## Export Screenshot / HTML

After finding the best view angle in the interactive window above, re-run the cell
with the same camera position to save a high-resolution PNG or standalone HTML.

In [ ]:
import os
os.makedirs("results/stage1/figures", exist_ok=True)

# Re-create plotter (must be off_screen for screenshot/export_html)
pl_export = render_interactive(gt_g, pred_g, params_g,
                               mode="intensity",
                               gt_thresh=0.05, pred_thresh=0.1)

# Optional: set camera position manually (retrieve from interactive pl.show() above)
# pl_export.camera_position = [(x, y, z), (cx, cy, cz), (ux, uy, uz)]

# --- Method 1: PNG screenshot ---
pl_export.off_screen = True
pl_export.show(auto_close=False)
pl_export.screenshot(
    "results/stage1/figures/gaussian_intensity_interactive.png",
    scale=2
)
print("PNG saved → results/stage1/figures/gaussian_intensity_interactive.png")

# --- Method 2: standalone HTML (for sharing / arxiv) ---
pl_export.export_html(
    "results/stage1/figures/gaussian_intensity_interactive.html"
)
print("HTML saved → results/stage1/figures/gaussian_intensity_interactive.html")
pl_export.close()

## Batch Process Multiple Samples

Uncomment and edit `representative` with actual sample IDs.

In [ ]:
# Representative samples (edit with your actual IDs from representative_samples.json)
# representative = [
#     "sample_0012",  # 1-foci shallow
#     "sample_0089",  # 1-foci deep
#     "sample_0156",  # 2-foci medium
#     "sample_0234",  # 2-foci deep
#     "sample_0345",  # 3-foci shallow
#     "sample_0467",  # 3-foci medium
# ]

# for sid in representative:
#     print(f"\n{'='*60}\nSample: {sid}")
#     gt_g, pred_g, params_g = predict_sample(samples_g / sid, model_g)
#     gt_u, pred_u, params_u = predict_sample(
#         samples_u / sid, model_u, normalize_gt=False
#     )
#     print(f"  G: gt_max={gt_g.max():.3f}, pred_max={pred_g.max():.3f}")
#     print(f"  U: gt_max={gt_u.max():.3f}, pred_max={pred_u.max():.3f}")
#     pl = render_interactive(gt_g, pred_g, params_g, mode="intensity")
#     pl.show()